# Figure 3b - three-cohort autoantibody heatmap by disease activity

In [ ]:
import os

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

DATA_DIR = "data"
RESULTS_DIR = "results"
OUT_DIR = f"{RESULTS_DIR}/fig"

UCSF_H5AD = f"{DATA_DIR}/adata_cohort1.h5ad"
ITN_H5AD = f"{DATA_DIR}/adata_cohort_2.h5ad"
NYU_H5AD = f"{DATA_DIR}/adata_cohort3.h5ad"
HIGHLOW_SUMMARY = (f"{RESULTS_DIR}/permutation_full/high_vs_low_mean/"
                   "highlow_mean_gene_level_clean_summary.csv")

QVAL_MAX = 0.1
TOP_N_GENES = 20

ROW_GROUPS = [
    ["NPIPB11", "NPIPB15", "NPIPB4", "NPIPB3", "LOC101060275"],
    ["H3F3C", "H3F3A"],
    ["LRFN5"],
    ["TACC2"],
    ["PPP1R12A"],
    ["SNRNP70"],
    ["ACIN1"],
    ["AGBL2"],
    ["SAFB"],
    ["AHDC1"],
    ["HEATR4"],
    ["HUWE1"],
    ["LHX9"],
    ["CACTIN"],
    ["CCAR1"],
]

HIGH_MIN_SLEDAI = 11
NONE_SLEDAI = 0
ITN_LABELS = {"high_disease_activity": "HIGH", "no_disease_activity": "NONE"}

ACTIVITY_ORDER = ["HIGH", "NONE"]
ACTIVITY_LABELS = {"HIGH": "High disease activity", "NONE": "No disease activity"}

THRESHOLD = 2.5
VMIN, VMAX = 0, 10
BIN_SIZE = 10
SORT_ZERO_THRESHOLD = 1.0
ROW_FONTSIZE = 12
FIG_WIDTH = 14
HIST_COLOR = "#C12026"
ACTIVITY_PALETTE = {"HIGH": "#4A2C76", "NONE": "#9B77B3"}
CENTER_PALETTE = {"UCSF": "#3654A5", "ITN": "#AFD3F0", "NYU": "#CBE1F5"}
CENTER_ORDER = ["UCSF", "ITN", "NYU"]
SEGMENT_GAP = 0.6
SEGMENT_MIN_WIDTH = 0.12

In [ ]:
def read_obs(path):
    adata = ad.read_h5ad(path, backed="r")
    obs = adata.obs.copy()
    adata.file.close()
    return obs


def read_peptides(path, layer, peptides):
    with h5py.File(path, "r") as handle:
        var_key = handle["var"].attrs.get("_index", "_index")
        var_key = var_key.decode() if isinstance(var_key, bytes) else var_key
        var_names = pd.Index(np.asarray(handle["var"][var_key]).astype(str))
        obs_key = handle["obs"].attrs.get("_index", "_index")
        obs_key = obs_key.decode() if isinstance(obs_key, bytes) else obs_key
        obs_names = pd.Index(np.asarray(handle["obs"][obs_key]).astype(str))

        missing = [p for p in peptides if p not in var_names]
        if missing:
            raise KeyError(f"{len(missing)} peptides absent from {path}: {missing[:3]}")

        positions = var_names.get_indexer(pd.Index(peptides))
        order = np.argsort(positions)
        values = handle["layers"][layer][:, positions[order]]

    matrix = pd.DataFrame(values, index=obs_names, columns=np.asarray(peptides)[order])
    return matrix.loc[:, list(peptides)]


def zscore_against_controls(matrix, control_mask):
    controls = matrix.loc[control_mask]
    return (matrix - controls.mean(axis=0)) / controls.std(axis=0, ddof=0)

In [ ]:
def select_genes():
    summary = pd.read_csv(HIGHLOW_SUMMARY, index_col=0)
    top = (summary[summary["fisher_qval"] <= QVAL_MAX]
           .sort_values("z_score", ascending=False)
           .head(TOP_N_GENES))

    wanted = [gene for group in ROW_GROUPS for gene in group]
    if set(wanted) != set(top.index):
        raise ValueError(
            "ROW_GROUPS does not match the selected genes; "
            f"only in groups: {sorted(set(wanted) - set(top.index))}; "
            f"only in selection: {sorted(set(top.index) - set(wanted))}"
        )
    return top.loc[wanted, "best_peptide"]


def sledai_activity(scores):
    s = pd.to_numeric(scores, errors="coerce")
    activity = pd.Series(pd.NA, index=s.index, dtype=object)
    activity[s >= HIGH_MIN_SLEDAI] = "HIGH"
    activity[s == NONE_SLEDAI] = "NONE"
    return activity


def one_per_patient(samples):
    n_states = samples.groupby("patient")["activity"].transform("nunique")
    excluded = samples.loc[n_states > 1, "patient"].nunique()
    kept = (samples[n_states == 1]
            .sort_values(["patient", "visit"], kind="mergesort")
            .groupby("patient", sort=False).head(1))
    return kept["activity"], excluded

In [ ]:
def ucsf_block(peptides):
    obs = read_obs(UCSF_H5AD)
    obs.loc[obs["case_control"].isna(), "case_control"] = "lupus"
    lupus = obs[obs["case_control"] == "lupus"]
    activity = sledai_activity(lupus["sledai_score"])
    lupus = lupus[activity.notna()]
    samples = pd.DataFrame({
        "patient": lupus["clues_cohort_id"].astype(str),
        "visit": pd.to_datetime(lupus["visit_date"].astype(str), format="%Y-%m-%d"),
        "activity": activity[lupus.index],
    })
    activity, excluded = one_per_patient(samples)
    zscores = read_peptides(UCSF_H5AD, "zscores_4andhalf", list(peptides))
    return zscores.loc[activity.index], activity, excluded


def itn_block(peptides):
    obs = read_obs(ITN_H5AD)
    label = obs["high_vs_no"].astype(str)
    lupus = obs[label.isin(ITN_LABELS)]
    activity = label[lupus.index].map(ITN_LABELS)
    if not (sledai_activity(lupus["sledai_score"]) == activity).all():
        raise ValueError("high_vs_no does not match the SLEDAI thresholds")

    samples = pd.DataFrame({
        "patient": lupus["sample_ID"].astype(str),
        "visit": pd.to_numeric(lupus["visit_num"]),
        "activity": activity,
    })
    activity, excluded = one_per_patient(samples)
    full = read_peptides(ITN_H5AD, "log_fold_change_over_AG", list(peptides)).loc[obs.index]
    zscores = zscore_against_controls(full, (obs["group"] == "healthy_control").values)
    return zscores.loc[activity.index], activity, excluded


def nyu_block(peptides):
    obs = read_obs(NYU_H5AD)
    lupus = obs[obs["group"] == "lupus"]
    activity = sledai_activity(lupus["sledai_score"])
    lupus = lupus[activity.notna()]
    patient_visit = lupus["subject_id"].astype(str).str.rsplit("_", n=1)
    samples = pd.DataFrame({
        "patient": patient_visit.str[0],
        "visit": pd.to_numeric(patient_visit.str[1]),
        "activity": activity[lupus.index],
    })
    activity, excluded = one_per_patient(samples)
    full = read_peptides(NYU_H5AD, "log_fold_change_over_AG", list(peptides)).loc[obs.index]
    zscores = zscore_against_controls(full, (obs["group"] == "control").values)
    return zscores.loc[activity.index], activity, excluded


def build_matrix(peptides):
    matrices, metas = [], []
    for center, loader in [("UCSF", ucsf_block), ("ITN", itn_block), ("NYU", nyu_block)]:
        zscores, activity, excluded = loader(peptides)
        print(f"{center}: {excluded} patients with both a high- and a no-activity sample excluded")
        zscores.columns = peptides.index
        matrices.append(zscores)
        metas.append(pd.DataFrame({"center": center, "activity": activity}))
    return pd.concat(matrices), pd.concat(metas)

In [ ]:
def order_samples(matrix, meta):
    keys = [group[0] for group in ROW_GROUPS if group[0] in matrix.columns]
    rank = pd.DataFrame(index=matrix.index)
    for gene in keys:
        z = matrix[gene]
        rank[gene] = np.where(z.abs() > SORT_ZERO_THRESHOLD, -z, np.inf)

    order = []
    for activity in ACTIVITY_ORDER:
        for center in CENTER_ORDER:
            block = meta.index[(meta["activity"] == activity) & (meta["center"] == center)]
            if len(block) == 0:
                continue
            order.extend(rank.loc[block].sort_values(by=keys, kind="mergesort").index.tolist())
    return matrix.loc[order], meta.loc[order]

In [ ]:
def threshold_colormap(threshold=THRESHOLD, vmin=VMIN, vmax=VMAX, n_colors=256):
    base = plt.colormaps["Reds"]
    cut = (threshold - vmin) / (vmax - vmin)
    colors = []
    for i in range(n_colors):
        position = i / (n_colors - 1)
        if position < cut:
            colors.append((1, 1, 1, 1))
        else:
            colors.append(base(0.25 + 0.75 * (position - cut) / (1 - cut)))
    cmap = LinearSegmentedColormap.from_list("white_to_deep_red", colors, N=n_colors)
    cmap.set_bad("white")
    return cmap


def contiguous_segments(values):
    values = np.asarray(values)
    if len(values) == 0:
        return []
    segments, start = [], 0
    for i in range(1, len(values)):
        if values[i] != values[start]:
            segments.append((values[start], start, i))
            start = i
    segments.append((values[start], start, len(values)))
    return segments


def _segment_span(start, end, gap=SEGMENT_GAP, min_width=SEGMENT_MIN_WIDTH):
    left, right = start + gap / 2.0, end - gap / 2.0
    if right > left:
        return left, right - left
    width = max((end - start) * 0.6, min_width)
    return start + max(((end - start) - width) / 2.0, 0), width


def draw_segments(ax, segments, palette, label=False):
    for name, start, end in segments:
        if name not in palette:
            continue
        x, width = _segment_span(start, end)
        ax.add_patch(Rectangle((x, 0), width, 1.0, facecolor=palette[name], edgecolor="none"))
        if label:
            ax.text(x + width / 2.0, 0.5, name, ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white")


def cohort_restart_bins(values, centers, bin_size=BIN_SIZE):
    positions, sums, widths = [], [], []
    for _, start, end in contiguous_segments(centers):
        edges = np.unique(np.append(np.arange(0, end - start, bin_size), end - start))
        for lo, hi in zip(edges[:-1], edges[1:]):
            positions.append(start + (lo + hi) / 2.0)
            sums.append(values[start + lo:start + hi].sum())
            widths.append(hi - lo)
    return np.array(positions), np.array(sums), np.array(widths)

In [ ]:
def plot_heatmap(matrix, meta, out_stem):
    rows = [gene for group in ROW_GROUPS for gene in group if gene in matrix.columns]
    values = matrix[rows].T
    displayed = values.mask(values < THRESHOLD)

    activity = meta["activity"].values
    centers = meta["center"].values
    n_total = values.shape[1]
    boundary = int((activity == "HIGH").sum())
    if not (activity[:boundary] == "HIGH").all():
        raise ValueError("samples are not ordered high-activity first")

    hits = (values >= THRESHOLD).sum(axis=0).values
    cmap = threshold_colormap()

    fig = plt.figure(figsize=(FIG_WIDTH, 0.25 * len(rows) + 2.6))
    grid = fig.add_gridspec(
        nrows=4, ncols=3,
        width_ratios=[boundary, 3, n_total - boundary],
        height_ratios=[0.18, 0.05, 0.05, 1],
        wspace=0.03, hspace=0.05,
    )
    axes = {name: (fig.add_subplot(grid[row, 0]), fig.add_subplot(grid[row, 2]))
            for row, name in enumerate(["hist", "center", "activity", "heatmap"])}
    colorbar_ax = inset_axes(axes["heatmap"][1], width="6%", height="70%", loc="center left",
                             bbox_to_anchor=(1.05, 0, 1, 1),
                             bbox_transform=axes["heatmap"][1].transAxes, borderpad=0)

    sides = [(0, slice(0, boundary), boundary, "HIGH"),
             (1, slice(boundary, n_total), n_total - boundary, "NONE")]

    histograms = []
    for side, window, width, _ in sides:
        positions, sums, widths = cohort_restart_bins(hits[window], centers[window])
        axes["hist"][side].bar(positions, sums, width=widths, color=HIST_COLOR,
                               edgecolor="white", linewidth=0.2)
        axes["hist"][side].set_xlim(0, width)
        histograms.append(sums)

    hist_max = max(h.max() for h in histograms) * 1.05
    for side, _, _, _ in sides:
        ax = axes["hist"][side]
        ax.set_ylim(0, hist_max)
        ax.tick_params(bottom=False, labelbottom=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    axes["hist"][0].set_ylabel(f"Σ(z≥{THRESHOLD})")

    for side, window, width, group in sides:
        for track, track_values, palette, label in [("center", centers, CENTER_PALETTE, True),
                                                    ("activity", activity, ACTIVITY_PALETTE, False)]:
            ax = axes[track][side]
            ax.axis("off")
            ax.set_ylim(0, 1)
            ax.set_xlim(0, width)
            draw_segments(ax, contiguous_segments(track_values[window]), palette, label=label)
        axes["activity"][side].text(width / 2, 0.5, ACTIVITY_LABELS[group], ha="center",
                                    va="center", color="white", weight="bold")

    sns.heatmap(displayed.iloc[:, :boundary], cmap=cmap, vmin=VMIN, vmax=VMAX,
                xticklabels=False, yticklabels=True, cbar=False, ax=axes["heatmap"][0])
    sns.heatmap(displayed.iloc[:, boundary:], cmap=cmap, vmin=VMIN, vmax=VMAX,
                xticklabels=False, yticklabels=False, cbar_ax=colorbar_ax,
                cbar_kws={"label": "Z-score"}, ax=axes["heatmap"][1])
    for ax in axes["heatmap"]:
        ax.set_xlabel("")
        ax.set_ylabel("")
    axes["heatmap"][0].set_yticklabels(axes["heatmap"][0].get_yticklabels(), fontsize=ROW_FONTSIZE)

    for track in axes.values():
        for ax in track:
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(f"{out_stem}.pdf", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

peptides = select_genes()
print(f"genes: {len(peptides)}")

matrix, meta = build_matrix(peptides)
matrix, meta = order_samples(matrix, meta)

print(meta.groupby(["activity", "center"], observed=True).size().to_string())
print(f"total patients {len(meta)}  (high {(meta['activity'] == 'HIGH').sum()} / "
      f"none {(meta['activity'] == 'NONE').sum()})")

plot_heatmap(matrix, meta, f"{OUT_DIR}/figure3b_three_cohort_heatmap")